# Dealer Gamma Hedging & GEX ในตลาด Bitcoin Options
### เคส Deribit quarterly expiry 26 มิ.ย. 2026 — max pain ~$72K แต่ปิดที่ ~$60K

Notebook ประกอบบทความเพจ **0DTE Labs** — คำนวณ **Gamma Exposure (GEX, ค่าประมาณแรง hedge ของ dealer)** ต่อ strike ของ BTC options แล้วดูว่าทำไม "กำแพง put $60K" ถึงทำตัวเป็น *ตัวเร่ง* ราคา ไม่ใช่ *แม่เหล็ก*

> ⚠️ **เนื้อหาเพื่อการศึกษาเท่านั้น ไม่ใช่คำแนะนำการลงทุน** — GEX เป็น *ค่าประมาณ (estimate)* จากสมมติฐานเรื่อง positioning ของ dealer ไม่ใช่ข้อเท็จจริงที่วัดได้ตรง ๆ
>
> **Dependencies:** `numpy`, `plotly` (รันบน Google Colab ได้ทันที)

ข้อมูลอ้างอิงจริง (Deribit, 26 มิ.ย. 2026): expiry $10.6B (~37% ของ OI), max pain ~$72K, settle ~$60K (~80% หมดค่า), 25-delta skew -10.7%/-11.3%/-9.6% (1d/7d/1m), put strike $60K > $1B notional


## 1. พื้นฐาน: delta, gamma และการ hedge ของ dealer

Market maker พยายามอยู่แบบ **delta-neutral** (พอร์ตไม่ได้/ไม่เสียเมื่อราคาขยับเล็กน้อย) แต่ **gamma** (อัตราที่ delta เปลี่ยนเมื่อ spot ขยับ) บังคับให้ต้อง re-hedge ตลอด

**Black-Scholes gamma:**

$$\Gamma = \frac{\phi(d_1)}{S\,\sigma\,\sqrt{T}}, \qquad d_1 = \frac{\ln(S/K) + (r + \tfrac{1}{2}\sigma^2)T}{\sigma\sqrt{T}}$$

- $\phi(\cdot)$ = ความหนาแน่นของการแจกแจงปกติมาตรฐาน — สูงสุดเมื่อ strike ใกล้ spot (ATM)
- แปลว่า **gamma กระจุกที่ ATM** และจางเมื่อ strike ไกลออกไป (deep OTM/ITM)

### ที่มาของสมการ (derivation)

เริ่มจากราคา European call ตาม Black & Scholes (1973):

$$C = S\,N(d_1) - K e^{-rT} N(d_2), \qquad d_2 = d_1 - \sigma\sqrt{T}$$

โดย $N(\cdot)$ คือ cumulative ของการแจกแจงปกติมาตรฐาน

**ขั้นที่ 1 — หา delta:** delta คืออนุพันธ์ของราคา option เทียบกับ spot

$$\Delta = \frac{\partial C}{\partial S} = N(d_1)$$

(การพิสูจน์ใช้เอกลักษณ์ $S\,\phi(d_1) = K e^{-rT}\phi(d_2)$ ทำให้เทอมที่เหลือจากการดิฟตัดกันหมด เหลือแค่ $N(d_1)$)

**ขั้นที่ 2 — หา gamma:** gamma คืออนุพันธ์ของ delta เทียบกับ spot อีกชั้น (อนุพันธ์อันดับสองของราคา option เทียบกับ spot)

$$\Gamma = \frac{\partial \Delta}{\partial S} = \phi(d_1)\cdot\frac{\partial d_1}{\partial S}$$

และเนื่องจาก $\frac{\partial d_1}{\partial S} = \frac{1}{S\sigma\sqrt{T}}$ จึงได้ $\Gamma = \frac{\phi(d_1)}{S\,\sigma\,\sqrt{T}}$

> 📌 **หมายเหตุ:** put กับ call มี gamma **เท่ากัน** (delta ของ put = $N(d_1)-1$ ดิฟแล้วได้ gamma ตัวเดียวกัน) จึงใช้ `bs_gamma` ตัวเดียวได้ทั้งสองฝั่ง
>
> **แหล่งอ้างอิง:** Black, F. & Scholes, M. (1973), "The Pricing of Options and Corporate Liabilities", *Journal of Political Economy* 81(3) · Hull, J. C., *Options, Futures, and Other Derivatives* (บท The Greek Letters)

In [ ]:
import numpy as np

def norm_pdf(x):
    """Standard normal pdf phi(x) = exp(-x^2/2)/sqrt(2*pi)."""
    return np.exp(-0.5 * x**2) / np.sqrt(2.0 * np.pi)

In [ ]:
def bs_gamma(S, K, T, sigma, r=0.0):
    """
    Black-Scholes gamma per 1 contract (1 unit underlying).
    Gamma = phi(d1) / (S * sigma * sqrt(T))
    """
    S = np.asarray(S, dtype=float)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return norm_pdf(d1) / (S * sigma * np.sqrt(T))

Ks = np.array([50000, 55000, 60000, 65000, 70000])
print("gamma per strike (spot=60k):", np.round(bs_gamma(60000.0, Ks, 7/365, 0.60), 8))

## 2. จำลอง option chain (อิงข้อมูลจริง Deribit 26 มิ.ย. 2026)

ใส่ **IV skew** (put ราคาแพงกว่า → IV สูงกว่า ตรงกับ 25-delta skew ที่ติดลบ) และ **OI** ที่ put กระจุกหนักที่ strike $60K (≈16,700 BTC ≈ $1.0B notional)


In [ ]:
# facts: max pain ~72k, settled ~60k, put wall at 60k > $1B notional
spot = 63000.0            # spot before settlement
T = 7 / 365               # ~7 days to expiry
r = 0.0
contract_size = 1.0       # Deribit BTC option = 1 BTC

strikes = np.arange(50000, 85001, 2500, dtype=float)

# IV skew: puts (low strikes) richer -> higher IV (matches negative 25d skew)
iv_put  = 0.75 - (strikes - 50000) / (85000 - 50000) * 0.25   # 0.75 -> 0.50
iv_call = 0.55 - (strikes - 50000) / (85000 - 50000) * 0.10   # 0.55 -> 0.45

# OI in contracts (=BTC). put wall at 60k = 16700 BTC ~ $1.0B
# strikes(k): 50 52.5 55 57.5 60 62.5 65 67.5 70 72.5 75 77.5 80 82.5 85
put_oi  = np.array([ 8000, 5000, 9000, 7000, 16700, 6000, 5000, 3000,
                     4000, 2500, 1500, 1000,  800,  500,  300], dtype=float)
call_oi = np.array([  200,  300,  400,  600,   900, 1500, 2500, 3500,
                     5000, 6000, 4500, 3500, 2500, 1500,  900], dtype=float)

assert len(strikes) == len(put_oi) == len(call_oi)
wall = put_oi[strikes == 60000][0]
print(f"put OI at 60k = {wall:.0f} BTC (~${wall*60000/1e9:.2f}B notional)")

## 3. GEX ต่อ strike + sign convention (จุดที่คนพลาดบ่อยที่สุด)

**Dollar-gamma ต่อการเคลื่อน 1% ของ spot** ต่อ strike:

$$\text{GEX}_K = \pm\,\Gamma_K \cdot OI_K \cdot \text{contract} \cdot S^2 \cdot 0.01$$

**เครื่องหมาย $\pm$ คือสมมติฐาน ไม่ใช่ข้อเท็จจริง** — ต้อง "เดา" ว่า dealer อยู่ฝั่งไหนของแต่ละ option ที่นี่ใช้ convention แบบ SqueezeMetrics: **dealer long calls (+), short puts (−)** ถ้ากลับสมมติฐานนี้ ภาพ GEX จะกลับด้านทันที — นี่คือ pitfall ที่ต้องระบุให้ชัดทุกครั้ง

- **Net GEX > 0** → dealer *long gamma* → เทรดสวนทาง (ซื้อตอนตก/ขายตอนขึ้น) → **หนีบราคา (pin)**
- **Net GEX < 0** → dealer *short gamma* → เทรดตามทาง (ขายตอนตก/ซื้อตอนขึ้น) → **เร่งราคา (accelerate)**


In [ ]:
def dollar_gamma(oi, iv):
    g = bs_gamma(spot, strikes, T, iv, r)
    return g * spot**2 * 0.01 * oi * contract_size

dg_call = dollar_gamma(call_oi, iv_call)
dg_put  = dollar_gamma(put_oi,  iv_put)

# *** SIGN CONVENTION = ASSUMPTION (not a fact) ***
# dealer LONG calls (+), SHORT puts (-). Flip it and the picture flips.
net_gex = dg_call - dg_put          # $ per 1% move
print(f"Total Net GEX = ${net_gex.sum()/1e6:.1f}M per 1% move")

In [ ]:
# gamma flip: where cumulative net GEX crosses zero
order = np.argsort(strikes)
cum = np.cumsum(net_gex[order])
flip_idx = np.where(np.diff(np.sign(cum)) != 0)[0]
if len(flip_idx):
    k0, k1 = strikes[order][flip_idx[0]], strikes[order][flip_idx[0]+1]
    print(f"Gamma flip between {k0:.0f} and {k1:.0f}")
else:
    print("No gamma flip in this strike range (net GEX same sign throughout)")

## 4. อ่านกราฟ GEX profile

แท่ง **แดง = โซน negative GEX (dealer short gamma → เร่ง)**, เขียว = positive (หนีบ) เส้นประคือ spot, put wall $60K และ max pain ~$72K


In [ ]:
import plotly.graph_objects as go

colors = ['crimson' if g < 0 else 'seagreen' for g in net_gex]
fig = go.Figure()
fig.add_bar(x=strikes, y=net_gex/1e6, marker_color=colors,
            hovertemplate='strike %{x:.0f}<br>GEX %{y:.1f} $M/1%%<extra></extra>',
            name='Net GEX')
fig.add_hline(y=0, line_width=1, line_color='black')
fig.add_vline(x=spot, line_dash='dash', line_color='black',
              annotation_text=f'spot {spot:.0f}', annotation_position='top')
fig.add_vline(x=60000, line_dash='dot', line_color='crimson',
              annotation_text='put wall 60k', annotation_position='bottom left')
fig.add_vline(x=72000, line_dash='dot', line_color='gray',
              annotation_text='max pain ~72k', annotation_position='top right')
fig.update_layout(
    title='BTC Net GEX profile (assumption: dealer long calls / short puts)',
    xaxis_title='Strike', yaxis_title='Net dealer GEX ($M / 1% move)',
    template='plotly_white', bargap=0.15, height=480)
fig.show()

## 5. หัวใจของเรื่อง: ทำไม max pain $72K แต่ปิด $60K

"ทฤษฎีแม่เหล็ก / max pain" บอกว่าราคาควรถูกดูดไปจุดที่ option หมดค่ามากสุด (~$72K) — **แต่ราคาปิดที่ $60K** ตรงข้ามกับที่ทฤษฎีทำนาย

คำอธิบายผ่าน GEX: โซนใกล้เงินถูกครอบด้วย **put OI มหาศาล** ที่ dealer เป็นฝั่ง short → **net GEX ติดลบ (short gamma)** เมื่อราคาไหลลงหา $60K, gamma พุ่ง, dealer ต้อง *ขาย* เพิ่มเพื่อ hedge → **เร่งขาลง** ไปหา strike ไม่ใช่ดูดขึ้น

Cell ล่างเป็นภาพประกอบเชิงกลไก (ไม่ใช่ backtest) ว่าทิศการ hedge ต่างกันอย่างไรระหว่าง long vs short gamma


In [ ]:
def hedge_flow(dealer_gamma_sign, price_moves):
    """long gamma -> trade against move (dampen); short gamma -> trade with move (amplify)."""
    return -dealer_gamma_sign * np.sign(price_moves)

moves = np.array([-3, -2, -1, +1, +2, +3], dtype=float)
print("price move (%):        ", moves)
print("long-gamma hedge flow: ", hedge_flow(+1, moves), " (against = pin)")
print("short-gamma hedge flow:", hedge_flow(-1, moves), " (with = accelerate)")

## 6. ข้อจำกัด & disclaimer

- **GEX เป็นค่าประมาณ** — เราไม่รู้ dealer positioning จริง ต้องสมมติ sign convention ทั้งหมด (คนละ convention → คนละภาพ)
- **crypto ≠ TradFi**: ตลาด 24/7, dealer hedge ด้วย perp/futures (มี funding), ไม่มี PFOF, retail สัดส่วนสูง → ผล hedge ไม่เหมือน SPX เป๊ะ
- **BS gamma** ในที่นี้ใช้แบบมาตรฐาน แต่ options คริปโตบน Deribit เป็น *inverse* (settle เป็น BTC) — ในงานจริงต้องปรับ ที่นี่ทำให้เข้าใจกลไกก่อน
- **max pain ≠ การปั่นราคา** และ **skew ≠ ทิศทางราคาที่แน่นอน**
- 📌 เนื้อหาเพื่อการศึกษา ไม่ใช่คำแนะนำการลงทุน

**References:** Glassnode Research (Taker-Flow GEX) · Glassnode Studio (BTC 25d skew, Deribit) · Deribit BTC Options Statistics · The Block (BTC option skew) · CoinDesk 30 มิ.ย. 2026
